In [ ]:
import pandas as pd
import numpy as np
from connect import connect_to_database
from retrieve_ids import get_ids_by_fields

In [ ]:
# read enrichment data
pathToEnrichmentData = ''
rich = pd.read_csv(pathToEnrichmentData,
                   
                   dtype={'id_number' : str, 'cellphone_number' : str})

rich['umuzi_email'] = rich['umuzi_email'].str.strip().str.lower()

In [ ]:
# read enrichment fields
pathToEnrichmentData2 = ''
richFields = pd.read_csv(
    pathToEnrichmentData2,
    dtype={
        'cellphone_number' : 'str',
        'id_number' : 'str'
    }
)

# normalize emails for easier time matching
richFields['email'] = richFields['email'].str.strip().str.lower()

In [ ]:
# read previous consolidated eos data
pathToPreviousMonthData = ''
previous = pd.read_csv(pathToPreviousMonthData)

In [ ]:
previous.head()

## Natalie/XPL EOs Data

In [ ]:
pathToXPLEos = ''

In [ ]:
# helper function to rename xpl columns
def generate_column_rename_map(
    total_columns: int,
    start_opportunity: int = 1
) -> dict:
    rename_map = {
        0: "umuzi_email",
        1: "cohort",
        2: "payment_1",
        3: "date_service_accessed_1",
        4: "earning_opportunity_extra_info_1",
        5: "earning_opportunity_type_1",
        6: "earning_opportunity_name_1",
    }

    col_idx = 7
    opportunity_num = 2

    while col_idx < total_columns:
        rename_map[col_idx] = f"payment_{opportunity_num}"
        rename_map[col_idx + 1] = f"date_service_accessed_{opportunity_num}"
        rename_map[col_idx + 2] = f"earning_opportunity_extra_info_{opportunity_num}"
        rename_map[col_idx + 3] = f"earning_opportunity_type_{opportunity_num}"
        rename_map[col_idx + 4] = f"earning_opportunity_name_{opportunity_num}"

        col_idx += 5
        opportunity_num += 1

    return rename_map

In [ ]:
# detect and parse dates appropriately
def parse_date(val):
    for fmt in ("%Y-%m-%d %H:%M:%S", "%d/%m/%Y", "%Y-%m-%d", "%d/%m/%Y"):
        try:
            return pd.to_datetime(val, format=fmt)
        except Exception:
            continue
    return pd.NaT  # or return val if you want to keep the original

In [ ]:
df1 = pd.read_excel(
    io=pathToXPLEos,
    sheet_name=0,
    skiprows=1,
    usecols=range(1,13)
)
df1.head()

In [ ]:
df1.shape

In [ ]:
cols = df1.columns.to_list()


firstMapper = generate_column_rename_map(df1.shape[1])

for idx, colName in firstMapper.items():
    cols[idx] = colName
    
df1.columns = cols

In [ ]:
date_cols = list(filter(lambda x: x.startswith('date'), df1.columns))
df1[date_cols] = df1[date_cols].map(parse_date)
    
df1.head()

In [ ]:
df2 = pd.read_excel(
    io=pathToXPLEos,
    sheet_name=1,
    skiprows=1,
    usecols=range(1,68)
)

df2

In [ ]:
secondMapper = generate_column_rename_map(total_columns=df2.shape[1])
cols2 = df2.columns.to_list()

for idx, colName in secondMapper.items():
    cols2[idx] = colName
    
df2.columns = cols2

df2 = df2[secondMapper.values()]

In [ ]:
date_cols = list(filter(lambda x: x.startswith('date'), df2.columns))
df2[date_cols] = df2[date_cols].map(parse_date)

In [ ]:
df3 = pd.read_excel(
    io=pathToXPLEos,
    sheet_name=2,
    skiprows=1,
    usecols=range(1,83)
)
df3.head()

In [ ]:
cols3 = df3.columns.to_list()
thirdMapper = generate_column_rename_map(df3.shape[1])

for idx, colName in thirdMapper.items():
    cols3[idx] = colName
    
df3.columns = cols3

df3 = df3[thirdMapper.values()]

In [ ]:
date_cols = list(filter(lambda x: x.startswith('date'), df3.columns))
df3[date_cols] = df3[date_cols].map(parse_date)

In [ ]:
df4 = pd.read_excel(
    io=pathToXPLEos,
    sheet_name=3,
    skiprows=1,
    usecols=range(1,63)
)
df4.head()

In [ ]:
cols4 = df4.columns.to_list()
fourthMapper = generate_column_rename_map(df4.shape[1])

for idx, colName in fourthMapper.items():
    cols4[idx] = colName
    
df4.columns = cols4

df4 = df4[fourthMapper.values()]

In [ ]:
# convert dates for df4
date_cols = list(filter(lambda x: x.startswith('date'), df4.columns))
df4[date_cols] = df4[date_cols].map(parse_date)

In [ ]:
df5 = pd.read_excel(
    io=pathToXPLEos,
    sheet_name=4,
    skiprows=1,
    usecols=range(1,38)
)
df5.head()

In [ ]:
cols5 = df5.columns.to_list()
fifthMapper = generate_column_rename_map(df5.shape[1])

for idx, colName in fifthMapper.items():
    cols5[idx] = colName
    
df5.columns = cols5

df5 = df5[fifthMapper.values()]

In [ ]:
date_cols = list(filter(lambda x: x.startswith('date'), df5.columns))
df5[date_cols] = df5[date_cols].map(parse_date)

In [ ]:
df1.shape, df2.shape, df3.shape, df4.shape, df5.shape

In [ ]:
# Ensure there aren't any blank rows sneaking in
for dataframe in [df1, df2, df3, df4, df5]:
    dataframe.dropna(subset='umuzi_email', inplace=True)

In [ ]:
# Blueprint for melting the dataframe to a long format
(pd.wide_to_long(
        df=df3,
        stubnames=[
            'earning_opportunity_type',
            'earning_opportunity_name', 
            'payment', 
            'date_service_accessed',
            'earning_opportunity_extra_info'
        ],
        i=['umuzi_email', 'cohort'],
        j='payment_number',
        sep="_",
        suffix="\\d+"
    )
        .reset_index()
        .dropna(subset=['payment'], how='any')
        .drop(columns='payment_number')
        .sort_values(by=['umuzi_email', 'date_service_accessed'])
        .reset_index(drop=True)
        .sample(10)
    )

In [ ]:
for dataframe in [df1, df2, df3, df4, df5]:
    result = (pd.wide_to_long(
        df=dataframe,
        stubnames=[
            'earning_opportunity_type',
            'earning_opportunity_name', 
            'payment', 
            'date_service_accessed',
            'earning_opportunity_extra_info'
        ],
        i=['umuzi_email', 'cohort'],
        j='payment_number',
        sep="_",
        suffix="\\d+"
    )
        .reset_index()
        .dropna(subset=['payment'], how='any')
        .drop(columns='payment_number')
        .sort_values(by=['umuzi_email', 'date_service_accessed'])
        .reset_index(drop=True)
    )
    
    # concatenate to data
    data = pd.DataFrame()
    data = pd.concat([data, result], ignore_index=True)

In [ ]:
data.sample(15)

In [ ]:
# drop any rows where date is null(means no payment was made)
data.dropna(subset=['date_service_accessed'], inplace=True)

In [ ]:
data.shape

In [ ]:
data.loc[data.duplicated(keep=False)]

In [ ]:
data['month'] = data['date_service_accessed'].dt.month_name()
data = data[previous.columns]
data.sample(10)

In [ ]:
data['umuzi_email'] = data['umuzi_email'].str.strip()

In [ ]:
set(data['umuzi_email']).difference(set(rich['umuzi_email']))

In [ ]:
data['payment'] = data['payment'].astype(float)

## Milestone/Mitchell/Finance Data

In [ ]:
pathToFinanceData = ''
milestone = pd.read_excel(
    pathToFinanceData,
    sheet_name=0,
    skiprows=1,
    dtype={'Umuzi Email Address OR ID #' : 'str'},
    usecols=range(1, 8)
)

milestone.head()

In [ ]:
milestone.columns

In [ ]:
column_mapping = {
    'Umuzi Email Address OR ID #': 'id_number',
    'Rand amount paid in a given month (can be zero for some gigs if learner on a stipend)': 'payment',
    "format DD/MM/YYYY - if only month and year known, put DD as 01 (if no date hasn't happened)": 'date_service_accessed',
    'If applicable - This is Useful for person compiing the data': 'cohort',
    'Useful for person compiing the data': 'earning_opportunity_extra_info',
    'type of earning opportunity accessed': 'earning_opportunity_name', 
    'if needed to differentiate within a type (we can add to this list if need be - just need alignment between parties)': 'earning_opportunity_type'
}

milestone.rename(columns=column_mapping, inplace=True)

In [ ]:
# drop na for fields that are relevant
milestone.dropna(subset=['id_number', 'date_service_accessed'], inplace=True)

In [ ]:
milestone = (
    milestone
    .merge(
        rich[['umuzi_email', 'id_number']],
        on='id_number',
        how='left'
    )
    .dropna(subset='umuzi_email')
    .drop(columns='id_number')
)

> Dropped the Free Lancer from Mitchell's Data

In [ ]:
milestone['month'] = pd.to_datetime(milestone['date_service_accessed'], dayfirst=True).dt.month_name()

milestone['date_service_accessed'] = pd.to_datetime(milestone['date_service_accessed'], dayfirst=True).dt.date

In [ ]:
milestone

## Partnerships

In [ ]:
pathToPartnershipsData = ''
partnerships = pd.DataFrame()

for sht_idx in range(0, 10):
    df = pd.read_excel(io=pathToPartnershipsData, sheet_name=sht_idx, skiprows=1,
                       usecols=range(1,8))
    
    # concatenate this to main dataframe
    partnerships = pd.concat([partnerships, df], ignore_index=True)


In [ ]:
cols = partnerships.columns.to_list()

# rename some columns
columnMapper = {
    0 : 'umuzi_email',
    1 : 'payment',
    2 : 'date_service_accessed',
    3 : 'cohort',
    4 : 'earning_opportunity_extra_info',
    5 : 'earning_opportunity_type',
    6 : 'earning_opportunity_name',
}

# data.rename(columns=columnMapper, inplace=True)
for idx, colName in columnMapper.items():
    cols[idx] = colName

partnerships.columns = cols


In [ ]:
# strip whitespace from string columns
stringCols = partnerships.select_dtypes(include='object').columns

partnerships[stringCols] = partnerships[stringCols].apply(lambda x: x.str.strip())

In [ ]:
partnerships['date_service_accessed'] = partnerships['date_service_accessed'].map(parse_date).dt.date
partnerships['month'] = pd.to_datetime(partnerships['date_service_accessed'], dayfirst=True).dt.month_name()

In [ ]:
# Remove any invalid entries
partnerships.dropna(subset=['date_service_accessed', 'payment'], inplace=True)

In [ ]:
# # drop rows where the date was not recorded due to transitions
partnerships.drop(
    labels=partnerships.loc[(partnerships['payment'].str.contains('Done')) | (partnerships['payment'].str.contains('Learner'))].index,
    axis=0,
    inplace=True
)

In [ ]:
# remove the rand symbol
partnerships['payment'] = (
    partnerships['payment'].str.replace('R', '')
    .str.replace(' ', '')
    .str.replace(',', '').astype('float')
    )

partnerships['umuzi_email'] = partnerships['umuzi_email'].str.strip()

In [ ]:
# correct a typo
partnerships['umuzi_email'] = partnerships['umuzi_email'].str.replace('feroza.chisty@umuzi.org', 'feroza.chishty@umuzi.org')

In [ ]:
set(partnerships['umuzi_email']) - set(rich['umuzi_email'])

In [ ]:
partnerships.dropna(subset=['earning_opportunity_type', 'earning_opportunity_name'], how='all', inplace=True)

## Launch Lab

In [ ]:
llab = pd.read_excel(
    io='',
    skiprows=1,
    usecols=[1, 2, 3, 4, 5, 6, 7],
    names = ['umuzi_email', 'payment', 'date_service_accessed', 'cohort',
             'earning_opportunity_extra_info', 'earning_opportunity_name', 'earning_opportunity_type']

)
llab.head()

In [ ]:
# remove the rand symbol
llab['payment'] = (llab['payment'].str.replace('R', '')
                   .str.replace(' ', '') # in between white spaces
                   .str.replace(',', '')
                   .str.strip()
                   .astype('float'))

llab['umuzi_email'] = llab['umuzi_email'].str.strip()

In [ ]:
set(llab['umuzi_email']) - set(rich['umuzi_email'])

In [ ]:
llab['date_service_accessed'] = pd.to_datetime(llab['date_service_accessed']).dt.date
llab['month'] = pd.to_datetime(llab['date_service_accessed']).dt.month_name()

In [ ]:
llab.sample(20)

## Community Team

In [ ]:
comm = pd.read_excel(
    io='',
    skiprows=1,
    usecols = range(1, 8),
    names=['umuzi_email', 'payment', 'date_service_accessed', 'cohort', 'earning_opportunity_extra_info',
           'earning_opportunity_type', 'earning_opportunity_name']
)
comm.head()

In [ ]:
# remove the rand symbol
# comm['payment'] = comm['payment'].str.replace('R', '').str.replace(' ', '').str.replace(',', '').astype('float')

comm['umuzi_email'] = comm['umuzi_email'].str.strip()

In [ ]:
# set(comm['umuzi_email']) - set(rich['umuzi_email'])

In [ ]:
comm['date_service_accessed'] = comm['date_service_accessed'].map(parse_date).dt.date

comm['month'] = pd.to_datetime(comm['date_service_accessed']).dt.month_name()

## SAP TEAM

In [ ]:
sap = pd.read_excel('',
            usecols=range(1, 8),
            skiprows=1,
            sheet_name=1,
            dtype={'Umuzi Email Address OR ID #': str, "Rand amount paid in a given month (can be zero for some gigs if learner on a stipend)": str}
            )
sap.head()

In [ ]:
sapColMapper = column_mapping.copy()
sapColMapper['Umuzi Email Address OR ID #'] = 'umuzi_email'

In [ ]:
# rename columns
sap.rename(columns=sapColMapper, inplace=True)

In [ ]:
sap.dropna(subset='umuzi_email', inplace=True)

In [ ]:
sap['umuzi_email'] = sap['umuzi_email'].str.rjust(13, '0')
sap['payment'] = sap['payment'].str.replace('R', '').str.replace(' ', '').str.replace(',', '').astype('float')

In [ ]:
sap.head()

In [ ]:
sap['date_service_accessed'] = sap['date_service_accessed'].map(parse_date).dt.date

sap['month'] = pd.to_datetime(sap['date_service_accessed']).dt.month_name()

In [ ]:
set(sap['umuzi_email']) - set(rich['id_number'])

In [ ]:
# set(sap['umuzi_email']) - set(rich['umuzi_email'])

In [ ]:
set(sap['umuzi_email']) - set(rich['email'])

In [ ]:
sap.shape[0]

In [ ]:
sap = (sap.merge(
    right=rich[['id_number', 'umuzi_email']],
    left_on='umuzi_email', right_on='id_number',
    how='left'
)
 [['umuzi_email_y', 'payment','date_service_accessed', 'cohort', 'earning_opportunity_extra_info', 'earning_opportunity_type', 'earning_opportunity_name', 'month']]
 .rename(columns={'umuzi_email_y': 'umuzi_email'})
 )

In [ ]:
sap.sample(10)

## Umuzi Interns

In [ ]:
interns = (
    pd.read_csv('',
                usecols=[1, 3, 4, 5, 6, 7, 9], skiprows=1,
                names=['umuzi_email', 'cohort', 'earning_opportunity_type', 'earning_opportunity_name', 
                       'earning_opportunity_extra_info', 'date_service_accessed', 'payment'])
)

In [ ]:
interns['date_service_accessed'] = pd.to_datetime(interns['date_service_accessed']).dt.date

interns['month'] = pd.to_datetime(interns['date_service_accessed']).dt.month_name()

In [ ]:
interns['payment'] = interns['payment'].str.replace('R', '').str.replace(" ", "").astype(float)

In [ ]:
interns.head()

## PUT IT ALL TOGETHER NOW

In [ ]:
combined = pd.concat([data, milestone, partnerships, 
                      llab, 
                    interns,
                    sap, previous, comm
                      ], ignore_index=True)

In [ ]:
combined.shape

In [ ]:
combined[combined.duplicated(keep=False)].sort_values(by=['umuzi_email', 'date_service_accessed'])

In [ ]:
combined.info()

In [ ]:
combined = combined.apply(lambda x: x.str.strip() if x.dtype=='object' else x)

In [ ]:
# check for duplicate payments
combined.duplicated().sum()

In [ ]:
combined.drop_duplicates(keep='first', inplace=True)

In [ ]:
combined.shape

In [ ]:
# Ad hoc adjustments for some people's emails

combined.loc[combined['umuzi_email']=='philip.mogale@umuzi.org', 'umuzi_email'] = 'phillip.mogale@umuzi.org'
combined.loc[combined['umuzi_email']=='nani.mlondolozi@umuzi.org', 'umuzi_email'] = 'mlondolozi.nani@umuzi.org'
combined.loc[combined['umuzi_email']=='ronaldbmorgan21@gmail.com', 'umuzi_email'] = 'ronald.bednico@umuzi.org'
combined.loc[combined['umuzi_email']=='dreadynerdy@gmail.com', 'umuzi_email'] = 'edmond.shange@umuzi.org'
combined.loc[combined['umuzi_email']=='devmfundo@gmail.com', 'umuzi_email'] = 'aaron.jiane@umuzi.org'

In [ ]:
# Naspers correction: buhle and co who had transitioned

combined.loc[2416, 'cohort'] = 'XH-PM-Apr-25'
combined.loc[2415, 'cohort'] = 'XH-PM-Apr-25'
combined.loc[2419, 'cohort'] = 'XH-PM-Apr-25'
combined.loc[2417, 'cohort'] = 'XH-PM-Apr-25'
combined.loc[2418, 'cohort'] = 'XH-PM-Apr-25'
combined.loc[2420, 'cohort'] = 'XH-PM-Apr-25'

In [ ]:
combined.drop(
    labels=[368, 372, 2862, 370, 371, 373, 2864, 369, 2859, 2861],
    axis=0, inplace=True
)

In [ ]:
# ad hoc corrections to fic gaps in earning opportunities

combined.loc[
    (combined['earning_opportunity_type'].isna())
    &
    (combined['earning_opportunity_name']=='Internship'),
    
    'earning_opportunity_type'
] = 'Placement'

In [ ]:
combined.loc[
    (combined['earning_opportunity_type'].isna())
    &
    (combined['earning_opportunity_name']!='Internship'),
    
    'earning_opportunity_type'
] = 'Experience gig'

## Output Sheets

### Cohort Remapping

In [ ]:
combined['cohort'].unique()

In [ ]:
remapCohort = {
    'XPL2' : 'XA-Jun-25',
    'XPL5' : 'XA-Nov-25',
    'XPL3' : 'XA-Aug-25',
    'XPL4' : 'XA-Sept-25',
    'C44 - BBD Learnership' : 'c44_wd',
    'XPL - Project Management' : 'XH-PMA-Apr-25',
    'XZ1 (Foundational Web Dev)': 'c46_wd',
    'XZ1 ( Foundational Web Dev)': 'c46_wd',
    'C19 - Grey Learnership': 'c19_cl',
    'C45 (Naspers Skills Programme)': 'c45_de',
    'VML_UI/UX' : 'XH-AUXUI-Aug-25',
    'XA1_AWD_2025' : 'c44_wd',
    'XM1_PM_2025': 'XH-PM-Apr-25',
    'XSD1_2025': 'XH-SD-Jun-25',
    'XZ1 (AWD)': 'a4_rn',
    'A3 0 BBD' : 'a3_rn',
    'C45 - Naspers 2024': 'c45_de',
    'C45' : 'c45_ux_s',
    'C39': 'c39_java',
    'EE SA_Adv_Web_Dev': 'XH-AWD-Aug-25'
}

# rename the cohorts to fit the slug names

combined['cohort'] = combined['cohort'].map(remapCohort)

In [ ]:
set(combined['cohort'].fillna("Missing Value"))

In [ ]:
conn = connect_to_database()

slugMap = get_ids_by_fields(
    fields={'slug', 'name'},
    table='programmes',
    conn=conn,
    filters={'slug': set(combined['cohort'].fillna("Missing Value"))} # type: ignore
)

slugMap = {entry['slug']: entry['name'] for entry in slugMap}


In [ ]:
slugMap

In [ ]:
set(combined['cohort'])

### Demographics Format

In [ ]:
neededCols = ['umuzi_email', 'cellphone_number','id_number', 'gender', 'race', 'residential_area_type',
  'province', 'nearest_metro', 'has_disability_or_differently_abled', 'date_of_birth',
  'cohort', 'earning_opportunity_extra_info', 'earning_opportunity_type', 'earning_opportunity_name', 'date_service_accessed',
  'first_name', 'last_name', 'payment', 'learner_id']

In [ ]:
rich_u = (
    rich
    .sort_values('learner_id')
    .drop_duplicates(subset='umuzi_email', keep='last')
)

richFields_email_u = (
    richFields
    .sort_values('learner_id')
    .drop_duplicates(subset='email', keep='last')
)

richFields_umuzi_u = (
    richFields
    .sort_values('learner_id')
    .drop_duplicates(subset='umuzi_email', keep='last')
)


assert rich_u['umuzi_email'].is_unique
assert richFields_email_u['email'].is_unique
assert richFields_umuzi_u['umuzi_email'].is_unique

In [ ]:
first_merge = combined.merge(
    rich_u[['umuzi_email', 'cellphone_number', 'id_number', 'race', 'gender', 'date_of_birth',
          'nearest_metro', 'has_disability_or_differently_abled', 'province', 'residential_area_type',
          'first_name', 'last_name', 'learner_id']],
    on='umuzi_email',
    how='left'
)

# Pickup any from Rich Fields whose email is not null and shared
second_merge = combined.merge(
    richFields_email_u[['email', 'cellphone_number', 'id_number', 'race', 'gender', 'date_of_birth',
                'nearest_metro', 'has_disability_or_differently_abled', 'province', 'residential_area_type',
                'first_name', 'last_name', 'learner_id']],
    left_on='umuzi_email', right_on='email',
    how='left'
)

# Pickup any from Rich Fields whose umuzi_email is not null
third_merge = combined.merge(
    richFields_umuzi_u[[ 'cellphone_number', 'id_number', 'race', 'gender', 'date_of_birth', 'umuzi_email',
                'nearest_metro', 'has_disability_or_differently_abled', 'province', 'residential_area_type',
                'first_name', 'last_name', 'learner_id']],
    on='umuzi_email',
    how='left')

demographics = (
    first_merge
    .combine_first(second_merge)
    .combine_first(third_merge)
    .drop_duplicates(subset=['umuzi_email', 'earning_opportunity_extra_info', 'earning_opportunity_type', 'earning_opportunity_name', 'date_service_accessed', 'payment'], keep='first')
    [neededCols]
    
    
)

demographics['learner_id'] = demographics['learner_id'].astype('Int64')

In [ ]:
combined.shape

> You should not have more rows in demographics/first_merge/second/third_merge than combined seeing as that would mean duplication on the email field

In [ ]:
first_merge.shape, second_merge.shape, third_merge.shape, demographics.shape

In [ ]:
(
    demographics.loc[demographics['cellphone_number'].isna(), 'umuzi_email']
    .pipe(lambda s: s[s.str.contains('umuzi', na=False)])
    .unique()
)

In [ ]:
demographics['earning_opportunity_type'].unique()

In [ ]:
# create new columns for age
# calculate age relevant details
demographics['age_service_accessed'] = (
    demographics['date_service_accessed'].map(parse_date)- demographics['date_of_birth'].map(parse_date)
).dt.days // 365

# add age bins
demographics['age_range'] = pd.cut(
    x=demographics['age_service_accessed'],
    bins=[-np.inf, 17, 25, 35, np.inf],
    labels=['17 and below', '18-25', '26-35', '36+']
)


demographics['month_of_service_accessed'] = demographics['date_service_accessed'].map(parse_date).dt.month_name()

demographics['month_of_service_accessed'] = (demographics['month_of_service_accessed'] +
                                             " " + 
                                             demographics['date_service_accessed'].map(parse_date).dt.year.astype(str))

demographics.sample(10)

### Remove Duplicates

In [ ]:
# drop duplicates before summarizations
combined.duplicated().sum()

In [ ]:
combined.loc[combined.duplicated(keep=False)].sort_values(by=['umuzi_email', 'date_service_accessed'])

In [ ]:
combined.drop_duplicates(subset=['umuzi_email', 'payment', 'date_service_accessed', 'earning_opportunity_name', 'earning_opportunity_extra_info', 'earning_opportunity_type'],
                         keep='first', inplace=True)

In [ ]:
combined.drop(
    labels=combined[combined['payment'] == 0].index,
    axis=0,
    inplace=True
)

In [ ]:
combined.drop(
    labels=combined.loc[combined['date_service_accessed'].isna()].index,
    axis=0,
    inplace=True
)

In [ ]:
combined.shape

In [ ]:
combined['umuzi_email'].str.strip().str.lower().unique().__len__()

> the followinge export becomes the "previous" dataframe in the next reporting cycle

In [ ]:
# export for next reporting cycle
# combined.to_csv('', index=False)

### Wide Format

In [ ]:
combined['row_id'] = combined.groupby(by=['umuzi_email', 'month']).cumcount()


wide = (
    combined
    .pivot(
        index=['umuzi_email', 'cohort','row_id'],
        columns="month",
        values=['payment', 'earning_opportunity_type', 'earning_opportunity_name', 'earning_opportunity_extra_info',]
))

wide.columns = [f"{month}_{col}" for col, month in wide.columns]
wide = wide.reset_index().drop(columns='row_id')

id_cols = ['umuzi_email', 'cohort']

# extract months dynamically from column names
months = sorted(
    {
        col.split('_')[0] for col in wide.columns[1:] if '_' in col
    },
    key = lambda m : pd.to_datetime(m, format='%B')
)

fields = [
    "payment",
    "earning_opportunity_type",
    "earning_opportunity_name",
    "earning_opportunity_extra_info"
]

ordered_cols = id_cols + [f"{m}_{field}" for m in months for field in fields]

wide = wide.reindex(columns=ordered_cols)

wide.head()

### Result Format

In [ ]:
result = (
    combined
    .groupby(by=['umuzi_email', 'month'], as_index=False)
    .agg(
        payment_sum=('payment', 'sum'),
        payment_count=('payment', 'count')
    )
    .pivot(
        index='umuzi_email', columns='month',
        values=['payment_sum', 'payment_count']
        )
    .fillna(0)
    .astype(int)
    .reset_index()
)

result.columns = ['umuzi_email']+ [
    f"{month} (Payment)" if metric == 'payment_sum'
    else f"{month} (Count)"
    for metric, month in result.columns[1:]
]


month_order = {
    'January': 13, 'February': 14, 'March': 15, 'April': 4,
    'May': 5, 'June': 6, 'July': 7, 'August': 8,
    'September': 9, 'October': 10, 'November': 11, 'December': 12
}

ordered_cols = ['umuzi_email'] + sorted(
    result.columns[1:],
    key=lambda c: month_order[c.split()[0]]  # take "April" from "April (Payment)"
)

result = result.reset_index()[ordered_cols]
result.head()

### Itumeleng Format

In [ ]:
eon_rank = (
    combined
    .groupby(['umuzi_email', 'earning_opportunity_type',
                'earning_opportunity_name'], as_index=False, dropna=False)
      .agg(first_eon_date=('date_service_accessed', 'min'))
)

eon_rank['op_num'] = (
    eon_rank
    .sort_values('first_eon_date')
    .groupby(['umuzi_email', 'earning_opportunity_type'])
    # ['first_eon_date']
    # .rank(method='first')
    .cumcount() + 1
)
eon_rank.head()

In [ ]:
pivotData =(combined
      .merge(
        eon_rank,
        on=['umuzi_email','earning_opportunity_type','earning_opportunity_name'],
        how='left'
    ))
pivotData['op_num'] = pivotData['op_num'].astype('Int64')

In [ ]:
#. Fill missing cohort information with nulls
pivotData['cohort'] = pivotData['cohort'].fillna('Missing Cohort Info')

In [ ]:
# Number of Gigs per Earning Opportunity Type

a = (
    pivotData.groupby(['umuzi_email','earning_opportunity_type', 'cohort', 'op_num', 'month'],
                as_index=False)
        .agg(amount=('payment','sum'),
             gigs=('payment','size'))
)

In [ ]:
# Add first date for each earning opportunity type secured
first_dates = (
    combined
    .groupby(['umuzi_email','earning_opportunity_type'], as_index=False)
    .agg(first_date=('date_service_accessed','min'))
)

a = a.merge(first_dates, 
                on=['umuzi_email','earning_opportunity_type'],
                how='left')


In [ ]:
pivotA = (
    a.pivot_table(
        index=['umuzi_email','first_date','earning_opportunity_type', 'cohort'],
        columns=['month', 'op_num'],
        values=['amount','gigs'],
        fill_value=0,
        aggfunc='sum'
    )
)

In [ ]:
# Sort index by month, metric then op_num
pivotA = pivotA.sort_index(axis=1, level=[1, 2, 0])


pivotA.columns = [
    (
        f"Earning Opportunity {op_num} ZAR Value ({month})"
        if metric == "amount"
        else f"# of Gig {op_num} ({month})"
    )
    for metric, month, op_num in pivotA.columns
]

pivotA = pivotA.reset_index()
pivotA.head()


In [ ]:
earning_opportunitues_cols = [c for c in pivotA.columns if c.startswith('Earning Opportunity ')]
gig_cols = [c for c in pivotA.columns if c.startswith('# of')]

# Define month order for chronological sorting
# year 1 runs from April to March
month_order = {
    'January': 13, 'February': 14, 'March': 15, 'April': 4, 'May': 5, 'June': 6,
    'July': 7, 'August': 8, 'September': 9, 'October': 10, 'November': 11, 'December': 12
}

def gig_sort_key(gig):
    # Extract the month from the string (it's in parentheses at the end)
    month = gig.split('(')[-1].replace(')', '').strip()
    return month_order[month]

def opp_sort_key(opportunity):
    # Extract the month from the string (it's in parentheses at the end)
    month = opportunity.split('(')[-1].replace(')', '').strip()
    
    # Extract the opportunity number
    # The pattern is "Earning Opportunity X ZAR Value" where X is the number
    parts = opportunity.split()
    opportunity_num = int(parts[2])  # The number is the 3rd word
    
    return (month_order[month], opportunity_num)


sorted_opportunties = sorted(earning_opportunitues_cols, key=opp_sort_key)
sorted_gigs = sorted(gig_cols, key=gig_sort_key)

In [ ]:
# 1. Clean demographics
demo = demographics.assign(
    umuzi_email = demographics['umuzi_email']
                    .str.strip()
                    .str.lower()
                    .str.normalize('NFKC')
).drop_duplicates(subset=['umuzi_email'])

# 2. Clean pivotA
pivotA_clean = pivotA.assign(
    umuzi_email = pivotA['umuzi_email']
                    .str.strip()
                    .str.lower()
                    .str.normalize('NFKC')
)


# 3. Merge
breakdown = (
    pivotA_clean
    .merge(
        demo[['umuzi_email', 'id_number', 'cellphone_number', 'nearest_metro', 'province', 'race', 'gender', 'first_name', 'last_name']],
        on='umuzi_email',
        how='left'
    )
)

breakdown['Full Name'] = breakdown['first_name'] + ' ' + breakdown['last_name']
breakdown.drop(columns=['first_name', 'last_name', 'umuzi_email'], inplace=True)


In [ ]:
assert breakdown[breakdown['Full Name'].isna()].empty, "Some records do not have names, replace with their emails"

In [ ]:
# add status
statusDict = {
    "Development pathway" : "Active",
    "Placement" : "Placement",
    "SA Local Placements": "Placement",
    "Impact gig": "Active",
    "Experience gig": "Active"
}

In [ ]:
breakdown['Programme'] = breakdown['cohort'].map(lambda x: slugMap.get(x, 'Not Provided'))

In [ ]:
breakdown['Current Status'] = breakdown['earning_opportunity_type'].map(lambda x: statusDict.get(x))

In [ ]:
columnOrder = ['Full Name', 'id_number', 'first_date', "Current Status", 'cellphone_number', 'nearest_metro', 'province',
               'cohort', 'Programme', 'race', 'gender', 'earning_opportunity_type']
columnOrder = columnOrder + [item for pair in zip(sorted_opportunties, sorted_gigs) for item in pair]

In [ ]:
set(breakdown['earning_opportunity_type'])

In [ ]:
breakdown = breakdown[columnOrder]
breakdown

In [ ]:
assert (set(breakdown.columns) - set(columnOrder)) == set()

In [ ]:
# separate tab for demographics
demo_columns = ['umuzi_email', 'cellphone_number', 'id_number', 'gender', 'race', 'residential_area_type', 'province', 'nearest_metro',
                'has_disability_or_differently_abled', 'date_of_birth']

learner_demographics = (
    result[['umuzi_email']]
    .merge(
        rich[demo_columns],
        on='umuzi_email',
        how='left'
    )
)

In [ ]:
columnsEops = ['umuzi_email', 'learner_id']

for col in demographics.columns:
    if col not in demo_columns + ['first_name', 'last_name', 'learner_id']:
        columnsEops.append(col)
        
singles = demographics[columnsEops]
singles = singles[(singles['date_service_accessed'] >= '2025-10-01') &
             (singles['date_service_accessed'] <= '2025-12-31')]
singles

In [ ]:
(combined.loc[
    combined.duplicated(
        subset=['umuzi_email', 'payment', 'date_service_accessed', 'earning_opportunity_extra_info', 'earning_opportunity_name'], keep=False
    )
].sort_values(by=['umuzi_email', 'date_service_accessed'])
 
 )

In [ ]:
demographics.shape, wide.shape, result.shape,  breakdown.shape, singles.shape, learner_demographics.shape

In [ ]:
singles.head(1)

In [ ]:
# positive means someone isn't appearing in the breakdowns; 
# negative means someone snuck in and isn't in the result/summarized figures
result['November (Payment)'].sum() - (breakdown['Earning Opportunity 1 ZAR Value (November)'].sum() 
 + breakdown['Earning Opportunity 2 ZAR Value (November)'].sum() 
 + breakdown['Earning Opportunity 3 ZAR Value (November)'].sum()
 + breakdown['Earning Opportunity 4 ZAR Value (November)'].sum()
 )

In [ ]:
# number of people matching
# same rules as above
result['November (Count)'].sum() - (breakdown['# of Gig 1 (November)'].sum() + 
                                    breakdown['# of Gig 2 (November)'].sum() + 
                                    breakdown['# of Gig 3 (November)'].sum() +
                                    breakdown['# of Gig 4 (November)'].sum() 
                                    )

In [ ]:
# # ## Final Exports

with pd.ExcelWriter(
    '',
    mode='a',
    if_sheet_exists='replace',
    engine='openpyxl'
) as writer:
    learner_demographics.to_excel(writer, index=False, sheet_name="Learner Demographics")
    singles.to_excel(writer, index=False, sheet_name='One Per Learner')
    wide.to_excel(writer, index=False, sheet_name='Monthly Entries Breakdown')
    result.to_excel(writer, index=False, sheet_name='Summarized Opportunities')
    breakdown.to_excel(writer, index=False, sheet_name='Earning Opportunities Secured')